# Enterprise Software Sales — Synthetic Data Generation

Generates a realistic, multi-table dataset spanning **24 months** for an interactive dashboard aimed at enterprise software sales leaders.

## Schema Overview

```
┌────────────────┐       ┌───────────────────┐       ┌──────────────┐
│    Accounts    │       │   Opportunities   │       │   Products   │
│ (PK: acct_id)  │◄──FK──│ (PK: opp_id)      │       │(PK: prod_id) │
│                │       │  FK: account_id   │       │              │
│  Segment       │       │  FK: rep_id       │       │  Name, Tier  │
│  Industry      │       │  Stage, ACV       │       │  List ACV    │
│  Region        │       │  Lead Source      │       │  Category    │
│  Employees     │       │  Close Date       │       └──────┬───────┘
│  ACV Target    │       │  has_promotion    │              │
└────────────────┘       └────────┬──────────┘              │
┌──────────────┐                  │                         │
│  Sales Reps  │◄──FK─────────────┤                         │
│ (PK: rep_id) │                  │                         │
│ Quota Attain │                  ▼                         │
└──────────────┘       ┌────────────────────┐               │
                       │  Sales Activities  │               │
                       │ (PK: activity_id)  │               │
                       │  FK: opp_id        │               │
                       │  FK: rep_id        │               │
                       │  Type, Duration    │               │
                       └────────────────────┘               │
                       ┌────────────────────┐               │
                       │Opportunity_Products│◄──FK──────────┘
                       │ (PK: opp_prod_id)  │
                       │  FK: opp_id        │
                       │  FK: product_id    │
                       │  Line ACV          │
                       └────────────────────┘
                       ┌────────────────────┐
                       │    Promotions      │
                       │ (PK: promotion_id) │
                       │  FK: opp_id        │
                       │  Type, had_effect  │
                       └────────────────────┘
```

### Tables

| Table | PK | FKs | Purpose |
|---|---|---|---|
| **accounts** | `account_id` | — | Firmographics: segment, industry, region, employee count, annual ACV target |
| **sales_reps** | `rep_id` | — | Rep name, team, hire date, quota‑attainment profile |
| **products** | `product_id` | — | 8 products inspired by a data & AI platform (Lakehouse, SQL Compute, ML Studio, etc.) |
| **opportunities** | `opportunity_id` | `account_id`, `rep_id` | Pipeline: ACV, stage, lead source, close date, promotion flag |
| **opportunity_products** | `opp_product_id` | `opportunity_id`, `product_id` | Line items enabling cross-sell / upsell analysis |
| **promotions** | `promotion_id` | `opportunity_id` | One-per-opportunity promotion: type (Discount, Delivery Support, Enablement), effect flag |
| **sales_activities** | `activity_id` | `opportunity_id`, `rep_id` | Emails, meetings, POC, and enablement sessions for activity-vs-win-rate analysis |

### Realism Levers

- **Seasonality** — Q4 (Oct-Dec) trough with ~30 % lower new-opp creation; Q1 budget-flush spike.
- **Rep Variance** — Quota attainment drawn from a beta distribution so some reps hit 120 %+ while others sit at 60 %.
- **Account ACV Targets** — Each account has an annual spend target derived from employee count and a segment-specific per-employee rate with log-normal noise.
- **One Rep per Account** — Every account is permanently assigned to a single sales rep (a rep can own many accounts).
- **Promotions** — ~10 % of opportunities receive a promotion (Discount, Delivery Support, or Enablement); ~65 % of promotions have a measurable effect on ACV, pipeline velocity, or win rate.
- **Non-uniform Distributions** — ACV is log-normal; activity counts are Poisson; POC durations are exponential.

## 1. Install Dependencies

In [ ]:
%pip install faker --quiet

In [ ]:
dbutils.library.restartPython()

## 2. Configuration

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from faker import Faker

CATALOG = "tabpfn_databricks"
SCHEMA = "agent"

SEED = 42
np.random.seed(SEED)
Faker.seed(SEED)
fake = Faker()

END_DATE = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
START_DATE = END_DATE - timedelta(days=730)

N_ACCOUNTS = 500
N_REPS = 30
N_OPPORTUNITIES = 4000
N_PRODUCTS = 8

print(f"Catalog: {CATALOG}")
print(f"Schema : {SCHEMA}")
print(f"Period : {START_DATE:%Y-%m-%d}  →  {END_DATE:%Y-%m-%d}")

## 3. Products Table

Static dimension — eight products modeled after a data & AI platform vendor (lakehouse, compute, ML, streaming, governance, GenAI, BI, support).

In [ ]:
products_pdf = pd.DataFrame([
    {"product_id": "PROD-001", "product_name": "Unified Lakehouse Platform", "tier": "Base",    "list_acv": 120000, "category": "Platform"},
    {"product_id": "PROD-002", "product_name": "Serverless SQL Compute",     "tier": "Add-on",  "list_acv": 48000,  "category": "Compute"},
    {"product_id": "PROD-003", "product_name": "ML Studio Pro",              "tier": "Premium", "list_acv": 72000,  "category": "AI/ML"},
    {"product_id": "PROD-004", "product_name": "Real-Time Streaming Engine", "tier": "Add-on",  "list_acv": 36000,  "category": "Data Engineering"},
    {"product_id": "PROD-005", "product_name": "Governance & Lineage Suite", "tier": "Add-on",  "list_acv": 30000,  "category": "Governance"},
    {"product_id": "PROD-006", "product_name": "GenAI Agent Builder",        "tier": "Premium", "list_acv": 60000,  "category": "AI/ML"},
    {"product_id": "PROD-007", "product_name": "BI Dashboards",              "tier": "Add-on",  "list_acv": 24000,  "category": "Analytics"},
    {"product_id": "PROD-008", "product_name": "Premium Support & TAM",      "tier": "Service", "list_acv": 36000,  "category": "Support"},
])

print(f"Products: {len(products_pdf)} rows")
products_pdf

## 4. Accounts Table

Three segments with distinct firmographic profiles. Each account gets an `annual_acv_target` representing its expected annual platform spend, derived from employee count and a segment-specific per-employee rate with log-normal noise.

| Segment | Share | Typical Employees | ACV / Employee | Typical ACV Target | Industries |
|---|---|---|---|---|---|
| High-Growth Tech | 30 % | 100 – 2 000 | $150 – $500 | $150 k – $800 k | SaaS, Fintech, Healthtech, AI/ML |
| Fortune 500 | 20 % | 10 000 – 250 000 | $8 – $25 | $300 k – $5 M | Financial Services, Healthcare, Manufacturing, Retail |
| Mid-Market | 50 % | 200 – 5 000 | $40 – $120 | $80 k – $400 k | Professional Services, Education, Media, Logistics |

In [ ]:
SEGMENT_PROFILES = {
    "High-Growth Tech": {
        "share": 0.30,
        "industries": ["SaaS", "Fintech", "Healthtech", "AI/ML", "Cybersecurity"],
        "emp_range": (100, 2000),
        "regions": {"North America": 0.50, "Europe": 0.25, "APAC": 0.20, "LATAM": 0.05},
        "acv_per_emp": (150, 500),
    },
    "Fortune 500": {
        "share": 0.20,
        "industries": ["Financial Services", "Healthcare", "Manufacturing", "Retail", "Energy"],
        "emp_range": (10000, 250000),
        "regions": {"North America": 0.45, "Europe": 0.30, "APAC": 0.20, "LATAM": 0.05},
        "acv_per_emp": (8, 25),
    },
    "Mid-Market": {
        "share": 0.50,
        "industries": ["Professional Services", "Education", "Media", "Logistics", "Real Estate"],
        "emp_range": (200, 5000),
        "regions": {"North America": 0.40, "Europe": 0.25, "APAC": 0.20, "LATAM": 0.15},
        "acv_per_emp": (40, 120),
    },
}

accounts_data = []
for i in range(N_ACCOUNTS):
    segment = np.random.choice(
        list(SEGMENT_PROFILES.keys()),
        p=[p["share"] for p in SEGMENT_PROFILES.values()],
    )
    profile = SEGMENT_PROFILES[segment]
    region = np.random.choice(
        list(profile["regions"].keys()),
        p=list(profile["regions"].values()),
    )
    lo, hi = profile["emp_range"]
    emp_count = int(np.random.lognormal(
        mean=np.log((lo + hi) / 3), sigma=0.6
    ))
    emp_count = max(lo, min(hi, emp_count))

    accounts_data.append({
        "account_id": f"ACCT-{i+1:04d}",
        "account_name": fake.company(),
        "segment": segment,
        "industry": np.random.choice(profile["industries"]),
        "employee_count": emp_count,
        "region": region,
        "annual_revenue_mm": round(emp_count * np.random.uniform(0.08, 0.25), 1),
        "annual_acv_target": int(round(
            emp_count * np.random.uniform(*profile["acv_per_emp"]) * np.random.lognormal(0, 0.25),
            -3,
        )),
        "created_date": fake.date_between(
            start_date=START_DATE - timedelta(days=365),
            end_date=END_DATE,
        ).isoformat(),
    })

accounts_pdf = pd.DataFrame(accounts_data)

print(f"Accounts: {len(accounts_pdf)} rows")
print(f"\nSegment distribution:\n{accounts_pdf['segment'].value_counts()}")
print(f"\nAnnual ACV target by segment (median):")
print(accounts_pdf.groupby("segment")["annual_acv_target"].median().apply(lambda x: f"${x:,.0f}").to_string())
accounts_pdf.head(10)

## 5. Sales Reps Table

Rep performance follows a **Beta distribution** (α=4, β=3) scaled to 40 %–140 % quota attainment, producing a realistic right-skewed spread where a handful of star performers coexist with a long tail of underperformers.

In [ ]:
TEAMS = ["Enterprise West", "Enterprise East", "Mid-Market", "Growth", "Strategic"]

reps_data = []
for i in range(N_REPS):
    quota_pct = np.random.beta(4, 3) * 100 + 40  # range ≈ 40 %–140 %
    reps_data.append({
        "rep_id": f"REP-{i+1:03d}",
        "rep_name": fake.name(),
        "team": np.random.choice(TEAMS),
        "hire_date": fake.date_between(
            start_date=START_DATE - timedelta(days=1095),
            end_date=END_DATE - timedelta(days=90),
        ).isoformat(),
        "annual_quota": int(np.random.choice([800000, 1000000, 1200000, 1500000])),
        "quota_attainment_pct": round(quota_pct, 1),
    })

reps_pdf = pd.DataFrame(reps_data)

print(f"Sales Reps: {len(reps_pdf)} rows")
print(f"Quota attainment — min: {reps_pdf['quota_attainment_pct'].min():.0f}%  "
      f"median: {reps_pdf['quota_attainment_pct'].median():.0f}%  "
      f"max: {reps_pdf['quota_attainment_pct'].max():.0f}%")
reps_pdf.head(10)

## 6. Opportunities & Promotions

Key realism features:

- **Seasonality** — Q4 (Oct–Dec) new-opp creation drops ~30 %; Q1 (Jan–Mar) spikes ~20 %.
- **Segment-dependent ACV** — Fortune 500 deals are 3–5× larger than Mid-Market.
- **Win rates by stage** — Discovery → Demo → Negotiation → Closed Won/Lost.
- **Rep skill** — higher-attainment reps get slightly better win rates and larger deals.
- **Promotions** — ~10 % of opportunities receive a promotion (one per opp, at most). Three types:

| Promotion Type | Effect when active |
|---|---|
| **Discount** | Reduces ACV by 10–25 %, boosts win rate by 5–12 pp |
| **Delivery Support** | Shortens pipeline by 15–30 %, boosts win rate by 3–5 pp |
| **Enablement** | Boosts win rate by 5–12 pp (strongest stage impact) |

~65 % of promotions produce a measurable effect; the rest are applied but show no impact.

In [ ]:
LEAD_SOURCES = ["Outbound", "Inbound", "Partner"]
LEAD_SOURCE_WEIGHTS = [0.40, 0.35, 0.25]

STAGES = ["Discovery", "Demo", "Negotiation", "Closed/Won", "Closed/Lost"]

ACV_PARAMS = {
    "High-Growth Tech": {"mu": 10.5, "sigma": 0.5},   # median ~$36 k
    "Fortune 500":      {"mu": 11.5, "sigma": 0.55},  # median ~$100 k
    "Mid-Market":       {"mu": 10.0, "sigma": 0.45},  # median ~$22 k
}

SEASONAL_MULTIPLIER = {
    1: 1.20, 2: 1.10, 3: 1.05,   # Q1 budget flush
    4: 1.00, 5: 1.00, 6: 0.95,   # Q2 steady
    7: 0.90, 8: 0.85, 9: 0.95,   # Q3 summer dip
    10: 0.75, 11: 0.70, 12: 0.65, # Q4 trough
}

PROMOTION_RATE = 0.10
PROMOTION_TYPES = ["Discount", "Delivery Support", "Enablement"]
PROMOTION_TYPE_WEIGHTS = [0.50, 0.25, 0.25]
PROMOTION_EFFECT_RATE = 0.65

account_ids = accounts_pdf["account_id"].tolist()
account_segment_map = dict(zip(accounts_pdf["account_id"], accounts_pdf["segment"]))

segment_weights = accounts_pdf["segment"].map(
    {"Fortune 500": 3.0, "High-Growth Tech": 2.0, "Mid-Market": 1.5}
)
account_weights = (segment_weights / segment_weights.sum()).tolist()

rep_ids = reps_pdf["rep_id"].tolist()
rep_quota_map = dict(zip(reps_pdf["rep_id"], reps_pdf["quota_attainment_pct"]))

# Each account is permanently assigned to exactly one rep (but a rep can own many accounts)
account_rep_map = {acct: np.random.choice(rep_ids) for acct in account_ids}


def pick_stage(rep_attainment_pct, lead_source, promo_boost=0.0):
    """Assign final stage with probabilities influenced by rep skill, lead source, and promotion."""
    base_win = 0.28
    rep_bonus = (rep_attainment_pct - 90) / 500  # ±0.10 swing
    source_bonus = {"Inbound": 0.05, "Partner": 0.03, "Outbound": -0.02}[lead_source]
    win_prob = np.clip(base_win + rep_bonus + source_bonus + promo_boost, 0.08, 0.55)

    r = np.random.random()
    if r < 0.12:
        return "Discovery"
    elif r < 0.28:
        return "Demo"
    elif r < 0.42:
        return "Negotiation"
    elif r < 0.42 + win_prob:
        return "Closed/Won"
    else:
        return "Closed/Lost"


opps_data = []
promotions_data = []
for i in range(N_OPPORTUNITIES):
    created = fake.date_between(start_date=START_DATE, end_date=END_DATE)
    month_mult = SEASONAL_MULTIPLIER[created.month]

    if np.random.random() > month_mult:
        continue

    acct_id = np.random.choice(account_ids, p=account_weights)
    segment = account_segment_map[acct_id]
    rep_id = account_rep_map[acct_id]
    lead_source = np.random.choice(LEAD_SOURCES, p=LEAD_SOURCE_WEIGHTS)

    acv_p = ACV_PARAMS[segment]
    raw_acv = np.random.lognormal(acv_p["mu"], acv_p["sigma"])
    acv = round(raw_acv, -2)

    days_to_close = int(np.random.exponential(scale=75)) + 14
    end_d = END_DATE.date() if isinstance(END_DATE, datetime) else END_DATE

    has_promotion = np.random.random() < PROMOTION_RATE
    promo_type = None
    promo_had_effect = False
    promo_boost = 0.0

    if has_promotion:
        promo_type = np.random.choice(PROMOTION_TYPES, p=PROMOTION_TYPE_WEIGHTS)
        promo_had_effect = np.random.random() < PROMOTION_EFFECT_RATE
        if promo_had_effect:
            if promo_type == "Discount":
                acv = round(acv * np.random.uniform(0.75, 0.90), -2)
                promo_boost = np.random.uniform(0.05, 0.12)
            elif promo_type == "Delivery Support":
                days_to_close = max(14, int(days_to_close * np.random.uniform(0.70, 0.85)))
                promo_boost = np.random.uniform(0.03, 0.05)
            elif promo_type == "Enablement":
                promo_boost = np.random.uniform(0.05, 0.12)

    stage = pick_stage(rep_quota_map[rep_id], lead_source, promo_boost)

    close_date = created + timedelta(days=days_to_close)
    if close_date > end_d + timedelta(days=180):
        close_date = end_d + timedelta(days=np.random.randint(30, 180))

    opp_id = f"OPP-{len(opps_data)+1:05d}"
    opps_data.append({
        "opportunity_id": opp_id,
        "account_id": acct_id,
        "rep_id": rep_id,
        "lead_source": lead_source,
        "acv": acv,
        "stage": stage,
        "created_date": created.isoformat(),
        "close_date": close_date.isoformat(),
        "days_in_pipeline": days_to_close,
        "has_promotion": has_promotion,
    })

    if has_promotion:
        promo_applied = fake.date_between(
            start_date=created,
            end_date=min(created + timedelta(days=max(1, days_to_close // 2)), end_d),
        )
        promotions_data.append({
            "promotion_id": f"PROMO-{len(promotions_data)+1:05d}",
            "opportunity_id": opp_id,
            "promotion_type": promo_type,
            "applied_date": promo_applied.isoformat(),
            "had_effect": promo_had_effect,
        })

opps_pdf = pd.DataFrame(opps_data)
promotions_pdf = pd.DataFrame(promotions_data)

print(f"Opportunities: {len(opps_pdf)} rows (after seasonal filtering)")
print(f"  with promotion: {opps_pdf['has_promotion'].sum()} ({opps_pdf['has_promotion'].mean()*100:.1f}%)")
print(f"\nStage distribution:\n{opps_pdf['stage'].value_counts()}")
print(f"\nMonthly volume (shows Q4 trough):")
opps_pdf["month"] = pd.to_datetime(opps_pdf["created_date"]).dt.to_period("M")
print(opps_pdf.groupby("month").size().tail(12).to_string())
opps_pdf.drop(columns=["month"], inplace=True)
opps_pdf.head(10)

### Promotions Summary

At most one promotion per opportunity. The `promotions` table records the type, application date, and whether the promotion had a measurable effect on the deal.

In [ ]:
print(f"Promotions: {len(promotions_pdf)} rows")
print(f"\nType distribution:\n{promotions_pdf['promotion_type'].value_counts()}")
print(f"\nEffect rate: {promotions_pdf['had_effect'].mean()*100:.1f}%")

promo_opps = opps_pdf[opps_pdf["has_promotion"]]
no_promo_opps = opps_pdf[~opps_pdf["has_promotion"]]

closed_promo = promo_opps[promo_opps["stage"].isin(["Closed/Won", "Closed/Lost"])]
closed_no_promo = no_promo_opps[no_promo_opps["stage"].isin(["Closed/Won", "Closed/Lost"])]

win_rate_promo = (closed_promo["stage"] == "Closed/Won").mean() if len(closed_promo) > 0 else 0
win_rate_no_promo = (closed_no_promo["stage"] == "Closed/Won").mean() if len(closed_no_promo) > 0 else 0

print(f"\nWin rate comparison:")
print(f"  With promotion:    {win_rate_promo:.1%}")
print(f"  Without promotion: {win_rate_no_promo:.1%}")

promotions_pdf.head(10)

## 7. Opportunity–Products Table (Line Items)

Each opportunity gets 1–4 product line items. Every opp includes **Core Platform**; add-ons are attached probabilistically based on segment.
Opportunities with an active **Discount** promotion receive an additional 5–15 pp discount on line items.
This powers cross-sell / upsell visualisation.

In [ ]:
product_ids = products_pdf["product_id"].tolist()
product_acv_map = dict(zip(products_pdf["product_id"], products_pdf["list_acv"]))

discount_promo_opp_ids = set(
    promotions_pdf.loc[
        (promotions_pdf["promotion_type"] == "Discount") & promotions_pdf["had_effect"],
        "opportunity_id",
    ]
)

ADDON_PROB_BY_SEGMENT = {
    "Fortune 500":      0.55,
    "High-Growth Tech": 0.40,
    "Mid-Market":       0.25,
}

opp_products_data = []
line_id = 0
for _, opp in opps_pdf.iterrows():
    segment = account_segment_map[opp["account_id"]]
    addon_prob = ADDON_PROB_BY_SEGMENT[segment]
    extra_discount = np.random.uniform(0.05, 0.15) if opp["opportunity_id"] in discount_promo_opp_ids else 0.0

    # Core Platform is always included
    discount = min(np.random.uniform(0.0, 0.20) + extra_discount, 0.40)
    line_id += 1
    opp_products_data.append({
        "opp_product_id": f"OPRD-{line_id:06d}",
        "opportunity_id": opp["opportunity_id"],
        "product_id": "PROD-001",
        "line_acv": round(product_acv_map["PROD-001"] * (1 - discount), -2),
        "discount_pct": round(discount * 100, 1),
    })

    # Probabilistic add-ons
    for pid in product_ids[1:]:
        if np.random.random() < addon_prob:
            discount = min(np.random.uniform(0.0, 0.15) + extra_discount, 0.35)
            line_id += 1
            opp_products_data.append({
                "opp_product_id": f"OPRD-{line_id:06d}",
                "opportunity_id": opp["opportunity_id"],
                "product_id": pid,
                "line_acv": round(product_acv_map[pid] * (1 - discount), -2),
                "discount_pct": round(discount * 100, 1),
            })

opp_products_pdf = pd.DataFrame(opp_products_data)

print(f"Opportunity–Products: {len(opp_products_pdf)} rows")
print(f"\nProducts per opportunity:")
print(opp_products_pdf.groupby("opportunity_id").size().describe())
opp_products_pdf.head(10)

## 8. Sales Activities Table

Four activity types per opportunity: **Emails**, **Meetings**, **POC** (Proof of Concept), and **Enablement** sessions.

Activity volume correlates with deal stage progression — opps that advance further get more touches. POC duration is only set for deals that reach Demo or beyond. Opportunities with an active **Enablement** promotion receive 1–3 additional enablement sessions (60–120 min).

In [ ]:
STAGE_ORDER = {"Discovery": 1, "Demo": 2, "Negotiation": 3, "Closed/Won": 4, "Closed/Lost": 4}
ACTIVITY_TYPES = ["Email", "Meeting", "POC", "Enablement"]

enablement_promo_opp_ids = set(
    promotions_pdf.loc[
        (promotions_pdf["promotion_type"] == "Enablement") & promotions_pdf["had_effect"],
        "opportunity_id",
    ]
)

activities_data = []
act_id = 0

end_date_d = END_DATE.date() if isinstance(END_DATE, datetime) else END_DATE

for _, opp in opps_pdf.iterrows():
    stage_depth = STAGE_ORDER[opp["stage"]]
    _raw_created = opp["created_date"]
    opp_created = datetime.strptime(_raw_created, "%Y-%m-%d").date() if isinstance(_raw_created, str) else _raw_created
    _raw_close = opp["close_date"]
    opp_close = datetime.strptime(_raw_close, "%Y-%m-%d").date() if isinstance(_raw_close, str) else _raw_close
    rep_attain = rep_quota_map[opp["rep_id"]]

    # Higher-attainment reps log slightly more activity
    rep_mult = 0.8 + (rep_attain / 200)

    # Emails: Poisson, mean scales with stage depth
    n_emails = int(np.random.poisson(lam=3 * stage_depth * rep_mult))
    for _ in range(n_emails):
        act_date = fake.date_between(start_date=opp_created, end_date=min(opp_close, end_date_d))
        act_id += 1
        activities_data.append({
            "activity_id": f"ACT-{act_id:07d}",
            "opportunity_id": opp["opportunity_id"],
            "rep_id": opp["rep_id"],
            "activity_type": "Email",
            "activity_date": act_date.isoformat(),
            "duration_minutes": None,
            "poc_days": None,
        })

    # Meetings: fewer, longer
    n_meetings = int(np.random.poisson(lam=1.5 * stage_depth * rep_mult))
    for _ in range(n_meetings):
        act_date = fake.date_between(start_date=opp_created, end_date=min(opp_close, end_date_d))
        duration = int(np.random.choice([15, 30, 45, 60], p=[0.10, 0.40, 0.30, 0.20]))
        act_id += 1
        activities_data.append({
            "activity_id": f"ACT-{act_id:07d}",
            "opportunity_id": opp["opportunity_id"],
            "rep_id": opp["rep_id"],
            "activity_type": "Meeting",
            "activity_date": act_date.isoformat(),
            "duration_minutes": duration,
            "poc_days": None,
        })

    # POC: only for deals that reached Demo or beyond
    if stage_depth >= 2 and np.random.random() < 0.45:
        poc_days = int(np.random.exponential(scale=14)) + 3
        poc_earliest = opp_created + timedelta(days=7)
        poc_latest = min(opp_close - timedelta(days=poc_days), end_date_d)
        if poc_latest < poc_earliest:
            poc_latest = poc_earliest
        poc_start = fake.date_between(start_date=poc_earliest, end_date=poc_latest)
        act_id += 1
        activities_data.append({
            "activity_id": f"ACT-{act_id:07d}",
            "opportunity_id": opp["opportunity_id"],
            "rep_id": opp["rep_id"],
            "activity_type": "POC",
            "activity_date": poc_start.isoformat(),
            "duration_minutes": None,
            "poc_days": poc_days,
        })

    # Enablement sessions: only for opps with an active Enablement promotion
    if opp["opportunity_id"] in enablement_promo_opp_ids:
        n_sessions = np.random.randint(1, 4)
        for _ in range(n_sessions):
            act_date = fake.date_between(start_date=opp_created, end_date=min(opp_close, end_date_d))
            duration = int(np.random.choice([60, 90, 120], p=[0.3, 0.5, 0.2]))
            act_id += 1
            activities_data.append({
                "activity_id": f"ACT-{act_id:07d}",
                "opportunity_id": opp["opportunity_id"],
                "rep_id": opp["rep_id"],
                "activity_type": "Enablement",
                "activity_date": act_date.isoformat(),
                "duration_minutes": duration,
                "poc_days": None,
            })

activities_pdf = pd.DataFrame(activities_data)

print(f"Sales Activities: {len(activities_pdf)} rows")
print(f"\nActivity type distribution:\n{activities_pdf['activity_type'].value_counts()}")
print(f"\nPOC duration stats (days):")
print(activities_pdf.loc[activities_pdf["activity_type"] == "POC", "poc_days"].describe())
activities_pdf.head(10)

## 9. Validation & Summary Statistics

In [ ]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

for name, df in [
    ("accounts", accounts_pdf),
    ("sales_reps", reps_pdf),
    ("products", products_pdf),
    ("opportunities", opps_pdf),
    ("promotions", promotions_pdf),
    ("opportunity_products", opp_products_pdf),
    ("sales_activities", activities_pdf),
]:
    print(f"\n  {name:25s}  {len(df):>8,} rows  ×  {len(df.columns)} cols")

# Referential integrity checks
print("\n" + "=" * 60)
print("REFERENTIAL INTEGRITY CHECKS")
print("=" * 60)

opp_acct_ok = opps_pdf["account_id"].isin(accounts_pdf["account_id"]).all()
opp_rep_ok = opps_pdf["rep_id"].isin(reps_pdf["rep_id"]).all()
oprd_opp_ok = opp_products_pdf["opportunity_id"].isin(opps_pdf["opportunity_id"]).all()
oprd_prod_ok = opp_products_pdf["product_id"].isin(products_pdf["product_id"]).all()
act_opp_ok = activities_pdf["opportunity_id"].isin(opps_pdf["opportunity_id"]).all()
act_rep_ok = activities_pdf["rep_id"].isin(reps_pdf["rep_id"]).all()
promo_opp_ok = promotions_pdf["opportunity_id"].isin(opps_pdf["opportunity_id"]).all()
one_promo_per_opp = promotions_pdf.groupby("opportunity_id").size().max() <= 1

one_rep_per_acct = opps_pdf.groupby("account_id")["rep_id"].nunique().max() == 1

for label, ok in [
    ("opportunities.account_id → accounts", opp_acct_ok),
    ("opportunities.rep_id → sales_reps", opp_rep_ok),
    ("opp_products.opportunity_id → opportunities", oprd_opp_ok),
    ("opp_products.product_id → products", oprd_prod_ok),
    ("activities.opportunity_id → opportunities", act_opp_ok),
    ("activities.rep_id → sales_reps", act_rep_ok),
    ("promotions.opportunity_id → opportunities", promo_opp_ok),
    ("one promotion per opportunity (at most)", one_promo_per_opp),
    ("one rep per account", one_rep_per_acct),
]:
    status = "✓" if ok else "✗ BROKEN"
    print(f"  {status}  {label}")

# Win-rate by lead source
print("\n" + "=" * 60)
print("WIN RATE BY LEAD SOURCE")
print("=" * 60)
closed = opps_pdf[opps_pdf["stage"].str.startswith("Closed")]
win_rate = closed.groupby("lead_source")["stage"].apply(
    lambda s: (s == "Closed/Won").mean()
).round(3)
print(win_rate.to_string())

# ACV by segment
print("\n" + "=" * 60)
print("MEDIAN ACV BY SEGMENT")
print("=" * 60)
merged = opps_pdf.merge(accounts_pdf[["account_id", "segment"]])
print(merged.groupby("segment")["acv"].median().to_string())

# Annual ACV target vs actual won ACV (annualized)
print("\n" + "=" * 60)
print("ACV TARGET vs ACTUAL WON (annualized, by segment)")
print("=" * 60)
won = opps_pdf[opps_pdf["stage"] == "Closed/Won"].copy()
won["close_year"] = pd.to_datetime(won["close_date"]).dt.year
won_annual = won.groupby("account_id")["acv"].sum() / 2  # 24-month data → annualize
acct_compare = accounts_pdf[["account_id", "segment", "annual_acv_target"]].merge(
    won_annual.rename("actual_annual_acv"), on="account_id", how="left"
).fillna(0)
summary = acct_compare.groupby("segment").agg(
    median_target=("annual_acv_target", "median"),
    median_actual=("actual_annual_acv", "median"),
).assign(ratio=lambda d: (d["median_actual"] / d["median_target"]).round(2))
print(summary.to_string())

# Promotion impact
print("\n" + "=" * 60)
print("PROMOTION IMPACT")
print("=" * 60)
print(f"  Total promotions:  {len(promotions_pdf)}")
print(f"  Effect rate:       {promotions_pdf['had_effect'].mean():.1%}")
promo_merged = opps_pdf.merge(promotions_pdf[["opportunity_id", "promotion_type", "had_effect"]], on="opportunity_id", how="left")
promo_closed = promo_merged[promo_merged["stage"].isin(["Closed/Won", "Closed/Lost"])]
promo_closed_with = promo_closed[promo_closed["has_promotion"]]
promo_closed_without = promo_closed[~promo_closed["has_promotion"]]
wr_with = (promo_closed_with["stage"] == "Closed/Won").mean() if len(promo_closed_with) > 0 else 0
wr_without = (promo_closed_without["stage"] == "Closed/Won").mean() if len(promo_closed_without) > 0 else 0
print(f"\n  Win rate with promotion:    {wr_with:.1%}")
print(f"  Win rate without promotion: {wr_without:.1%}")
print(f"\n  Win rate by promotion type:")
for ptype in PROMOTION_TYPES:
    subset = promo_closed[promo_closed["promotion_type"] == ptype]
    wr = (subset["stage"] == "Closed/Won").mean() if len(subset) > 0 else 0
    print(f"    {ptype:20s}  {wr:.1%}  (n={len(subset)})")

## 10. Write to Delta Tables

Creates the schema and writes all seven tables as managed Delta tables in Unity Catalog.

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

TABLES = [
    ("accounts", accounts_pdf),
    ("sales_reps", reps_pdf),
    ("products", products_pdf),
    ("opportunities", opps_pdf),
    ("promotions", promotions_pdf),
    ("opportunity_products", opp_products_pdf),
    ("sales_activities", activities_pdf),
]

for name, pdf in TABLES:
    fqn = f"{CATALOG}.{SCHEMA}.{name}"
    sdf = spark.createDataFrame(pdf)
    sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(fqn)
    count = spark.table(fqn).count()
    print(f"  {fqn:55s}  {count:>8,} rows")

print("\nDone — all tables written as Delta.")

## 11. Sample Data Preview (5–10 rows per table)

Inline CSV previews for quick reference.

In [ ]:
SAMPLE_N = 8

for name, df in [
    ("accounts", accounts_pdf),
    ("sales_reps", reps_pdf),
    ("products", products_pdf),
    ("opportunities", opps_pdf),
    ("promotions", promotions_pdf),
    ("opportunity_products", opp_products_pdf),
    ("sales_activities", activities_pdf),
]:
    print(f"\n{'='*80}")
    print(f"  {name.upper()}  (first {SAMPLE_N} rows as CSV)")
    print(f"{'='*80}")
    print(df.head(SAMPLE_N).to_csv(index=False))